# 02 — モデリングと評価

**問い**: 東京都の公示地価について、**翌年の平米単価**をどこまで当てられるか。
そして「当たっている」という結論は、どこまで信用できるのか。

この notebook は CLI パイプライン（`python -m src.pipeline --stage all`）と
同じ `src/` の関数を呼びます。数値は `reports/metrics.json` と一致します。


## 0. セットアップ


In [1]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Notebooks live one level below the repository root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import cleaning, error_analysis, evaluate, features, plots
from src.config import Config, configure_matplotlib, quiet_lightgbm
from src.data_loader import load_all_raw, summarise

logging.basicConfig(level=logging.WARNING)
cfg = Config.load()
configure_matplotlib(cfg)
quiet_lightgbm()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

print(f"repository : {REPO_ROOT.name}")
print(f"config     : {cfg.path.name}")
print(f"seed       : {cfg.seed}")
from IPython.display import Image, display, display as display_image

print(f"train year : {cfg.split['train_year']}  ->  predict year: {cfg.split['predict_year']}")

from src import models as model_lib
from src.models import MODEL_ORDER

# 両 notebook は独立して実行できるようにしてある。01_eda.ipynb と同じ
# クリーニングを通したフレームをここでも用意する（コードは src/ を共用）。
raw = load_all_raw(cfg)
frames = {}
for _name, _frame in raw.items():
    _clean, _ = cleaning.clean_frame(_frame, source=_name)
    _built = cleaning.add_unit_price(features.build_features(_clean))
    _built["_site_key"] = features.site_key(_built)
    frames[_name] = _built

kouji_r7, kouji_r8 = frames["kouji_r7"], frames["kouji_r8"]
kijun_r6, kijun_r7 = frames["kijun_r6"], frames["kijun_r7"]
print({name: len(frame) for name, frame in frames.items()})


repository : tokyo-land-price-ml
config     : config.yaml
seed       : 42
train year : kouji_r7  ->  predict year: kouji_r8


{'kouji_r7': 2560, 'kouji_r8': 2560, 'kijun_r6': 1277, 'kijun_r7': 1280}


## 1. 分割の設計（ここが評価の信頼性を決める）

```
train : 地価公示 令和7年 (2,560 地点)   ← 学習。価格水準も含む
val   : 地価公示 令和8年 (1,280 地点)   ← 翌年を予測（モデル選択用）
test  : 地価公示 令和8年 (1,280 地点)   ← 翌年を予測（最終評価、val と排他）
```

**なぜランダム分割にしないのか**を、この notebook の最後で実測して示します。


In [2]:
from src import pipeline

datasets = pipeline.load_datasets(cfg)
train, val, test, split_meta = pipeline.build_split(cfg, datasets)
print(json.dumps(split_meta, ensure_ascii=False, indent=2, default=str) if (json := __import__("json")) else "")


{
  "train_source": "kouji_r7",
  "predict_source": "kouji_r8",
  "n_train": 2560,
  "n_val": 1280,
  "n_test": 1280,
  "train_year": [
    2025
  ],
  "predict_year": [
    2026
  ],
  "sites_in_both_years": 2518,
  "overlap_val_test": 0,
  "lag_coverage_train": 0.98359375,
  "lag_coverage_predict": 0.98359375,
  "n_train_with_lag": 2518,
  "kijun": {
    "train_year": "kijun_r7",
    "test_year": "kijun_r6",
    "n_train": 1280,
    "n_test": 1277,
    "rows": 1280.0,
    "lag_available": 1277.0,
    "lag_coverage": 0.99765625
  }
}


In [3]:
# 地点キーが分割をまたいで重複していないことを確認（同一地点の記憶を防ぐ）
train_keys, val_keys, test_keys = (set(f[["admin_code", "use_code", "seq_no"]].astype(str).agg("|".join, axis=1))
                                   for f in (train, val, test))
print(f"train ∩ val  = {len(train_keys & val_keys)}")
print(f"train ∩ test = {len(train_keys & test_keys)}")
print(f"val   ∩ test = {len(val_keys & test_keys)}")
print()
print("lag_unit_price のカバレッジ")
for label, frame in (("train(令和7年)", train), ("test(令和8年)", test)):
    print(f"  {label}: {frame['lag_unit_price'].notna().mean()*100:.1f}%")


train ∩ val  = 1256
train ∩ test = 1262
val   ∩ test = 0

lag_unit_price のカバレッジ
  train(令和7年): 98.4%
  test(令和8年): 98.6%


## 2. 学習に使う特徴量（リーク防止の確認）

`features.feature_columns()` がモデルに入れてよい列の唯一の定義です。


In [4]:
allowlist = features.feature_columns(cfg)
print(f"既定の特徴量: {len(allowlist)} 列")
for column in allowlist:
    print("  -", column)
print()
print("リーク列（既定では絶対に入らない）:", list(features.LEAKY_FEATURES))
print()
raw_price_columns = {"price_current", "price_previous", "unit_price", "prev_unit_price"}
print("生の価格列が特徴量に含まれていないか:",
      "OK" if raw_price_columns.isdisjoint(allowlist) else "NG")


既定の特徴量: 24 列
  - site_area
  - frontage_ratio
  - depth_ratio
  - road_width
  - station_distance
  - bcr
  - far
  - floors_total
  - far_bcr_ratio
  - municipality
  - use_district
  - current_use
  - structure
  - road_type
  - road_direction
  - fire_zone
  - nearest_station
  - locality
  - pref_zone_1
  - pref_zone_2
  - site_area_band
  - station_distance_band
  - road_width_band
  - lag_unit_price

リーク列（既定では絶対に入らない）: ['prev_unit_price', 'change_rate_prior_year']

生の価格列が特徴量に含まれていないか: OK


## 3. ベースラインとモデルの比較


In [5]:
runs = pipeline.run_models(cfg, train, val, test)
table = evaluate.metrics_frame({name: run.metrics for name, run in runs.items()})
table = table.pivot(index="model", columns="split", values=["mae", "rmse", "r2"])
table = table.reindex(MODEL_ORDER)
table.round(3)


mae                           rmse                           r2              
split                  test     train       val       test      train        val   test  train    val
model                                                                                                
median             4911.128  4212.389  4704.651  12610.181  10510.466  11087.438 -0.081 -0.068 -0.093
mean               5452.439  4927.667  5240.253  12153.947  10168.371  10621.211 -0.004  0.000 -0.003
lag                 749.251   724.503   711.710   1780.341   1673.606   1668.652  0.978  0.973  0.975
lag_drift           477.140  1041.900   450.843   1345.599   2252.472   1269.087  0.988  0.951  0.986
ridge              1626.983  1447.646  1449.256   6329.329   7281.286   4350.235  0.728  0.487  0.832
random_forest      1281.629   114.124  1242.251   2727.343    530.745   2619.823  0.949  0.997  0.939
gradient_boosting  1287.484    88.480  1258.321   2764.786    167.479   2746.992  0.948  1.000  0.933
lightgbm           1347.595   382.823  1307.305   3551.853   2255.797   3265.792  0.914  0.951  0.905

In [6]:
comparison = pd.DataFrame(
    {
        "train MAE": [runs[n].metrics["train"]["mae"] for n in MODEL_ORDER],
        "val MAE": [runs[n].metrics["val"]["mae"] for n in MODEL_ORDER],
        "test MAE": [runs[n].metrics["test"]["mae"] for n in MODEL_ORDER],
        "test R2": [runs[n].metrics["test"]["r2"] for n in MODEL_ORDER],
        "test/train gap": [runs[n].metrics["test"]["mae"] / runs[n].metrics["train"]["mae"] for n in MODEL_ORDER],
    },
    index=MODEL_ORDER,
)
comparison.round(2)


,train MAE,val MAE,test MAE,test R2,test/train gap
median,4212.39,4704.65,4911.13,-0.08,1.17
mean,4927.67,5240.25,5452.44,-0.00,1.11
lag,724.50,711.71,749.25,0.98,1.03
lag_drift,1041.90,450.84,477.14,0.99,0.46
ridge,1447.65,1449.26,1626.98,0.73,1.12
random_forest,114.12,1242.25,1281.63,0.95,11.23
gradient_boosting,88.48,1258.32,1287.48,0.95,14.55
lightgbm,382.82,1307.31,1347.59,0.91,3.52


### 結果の読み方

- **median / mean（定数予測）**: MAE 約 4,900〜5,500 円/㎡。R² はほぼ 0 か負。
  これが「何も学習しない」水準です。
- **lag（前年単価をそのまま予測）**: MAE 約 750 円/㎡。**最も強いベースライン**です。
  地価は1年ではほとんど動かないため、これは当然の結果であり、
  「機械学習がこのベースラインを超えられるか」が本当の勝負になります。
- **ridge**: 線形モデルでも 1,600 円/㎡ 程度まで下がります。
- **random_forest / gradient_boosting / lightgbm**: 1,280〜1,350 円/㎡。
  train MAE は 88〜385 円/㎡ と極端に小さく、**過学習が明確**です。


In [7]:
path = plots.plot_model_comparison(
    evaluate.metrics_frame({n: runs[n].metrics for n in MODEL_ORDER}),
    cfg.figures_dir / "06_model_comparison.png",
)
plt.close("all")
display_image(filename=str(path))


**この図から読み取れること**: val と test の MAE がほぼ同じで、
分割が安定していることが分かります（val と test は同じ年の別地点）。
勾配ブースティング系は線形モデル（ridge）より 20% ほど良いものの、
前年単価ベースライン（lag）には遠く及びません。


## 4. 過学習と汎化の確認（train / val / test の乖離）


In [8]:
gap = pd.DataFrame(
    {
        "train MAE": [runs[n].metrics["train"]["mae"] for n in MODEL_ORDER],
        "test MAE": [runs[n].metrics["test"]["mae"] for n in MODEL_ORDER],
    },
    index=MODEL_ORDER,
)
gap["test/train"] = (gap["test MAE"] / gap["train MAE"]).round(2)
gap["CV MAE (train年5-fold)"] = [runs[n].cv.get("cv_mae_mean", float("nan")) for n in MODEL_ORDER]
gap["CV std"] = [runs[n].cv.get("cv_mae_std", float("nan")) for n in MODEL_ORDER]
gap.round(1)


,train MAE,test MAE,test/train,CV MAE (train年5-fold),CV std
median,4212.4,4911.1,1.2,4212.8,510.4
mean,4927.7,5452.4,1.1,4927.7,481.9
lag,724.5,749.3,1.0,NaN,NaN
lag_drift,1041.9,477.1,0.5,1041.9,108.3
ridge,1447.6,1627.0,1.1,1642.4,513.6
random_forest,114.1,1281.6,11.2,270.1,68.9
gradient_boosting,88.5,1287.5,14.6,276.7,53.7
lightgbm,382.8,1347.6,3.5,627.9,211.4


### 確認できたこと

- 決定木系は **train MAE が test MAE の 1/10 以下**（gap 10〜15倍）。
  深さを制限していない `random_forest` が最も極端です。
- `ridge` の gap は 1.13 倍で、過学習はほとんどありません。
- 訓練年での 5-fold CV（MAE の標準偏差）はモデル間の相対的な安定性を示します。

地価データは「同じ場所は似た価格」という性質が極めて強いため、
決定木は学習地点を丸暗記しやすく、**地点を分けた瞬間に精度が落ちます**。
これがこの課題の本質的な難しさです。


## 5. 特徴量重要度


In [9]:
importance = runs["lightgbm"].importance
display(importance.round(4))
path = plots.plot_feature_importance(importance, cfg.figures_dir / "05_feature_importance.png")
plt.close("all")
display_image(filename=str(path))


,feature,importance,importance_share
0,lag_unit_price,1071.0,0.3570
1,site_area,315.0,0.1050
2,road_width,231.0,0.0770
3,far,226.0,0.0753
4,station_distance,203.0,0.0677
5,nearest_station,173.0,0.0577
6,municipality,162.0,0.0540
7,current_use,103.0,0.0343
8,floors_total,86.0,0.0287
9,structure,64.0,0.0213


**この図から読み取れること**: 前年単価（`num_lag_unit_price`）が圧倒的首位（シェア約 37%）で、
次に地積・容積率・前面道路幅員・駅距離が続きます。
つまり**「去年の価格」＋「土地の物理的属性」**でほぼ説明でき、
残差に効くのは都市計画（容積率）と立地（駅距離）です。


## 6. 予測 vs 実測 / 残差


In [10]:
best_name = min((n for n in MODEL_ORDER if n != "lag"),
                key=lambda n: runs[n].metrics["val"]["mae"])
print("val MAE 最小のモデル:", best_name)

merged = pipeline.merge_test_predictions(runs[best_name], test)
path = plots.plot_pred_vs_actual(merged, cfg.figures_dir / "03_pred_vs_actual.png", title=f"test, {best_name}")
plt.close("all")
display_image(filename=str(path))
path = plots.plot_residuals(merged, cfg.figures_dir / "04_residuals.png")
plt.close("all")
display_image(filename=str(path))


val MAE 最小のモデル: lag_drift


**この図から読み取れること**: 線形スケールでは低価格帯に点が密集し、
高価格帯（10万円/㎡超）で過小予測（点が y=x より下）が目立ちます。
対数スケールでは全体にばらつきが均一になり、**相対誤差で見れば
価格帯によらず同程度の精度**であることが分かります。
残差は 0 を中心に分布しますが、右裾が長く（大きな過小予測がある）、
これは都心の高額地点を取り切れていないことを示します。


## 7. 誤差分析


In [11]:
from src import error_analysis

by_use = error_analysis.group_summary(merged, "use_district_group")
display(by_use.round(0))


,use_district_group,n,mae,rmse,bias,median_abs_pct_error,mean_abs_pct_error
0,商業系,397,1176.0,2305.0,-1136.0,7.0,8.0
1,住居系(中高層),317,245.0,648.0,-201.0,3.0,5.0
2,工業系,114,179.0,472.0,-168.0,3.0,7.0
3,住居系(低層),438,105.0,338.0,-28.0,2.0,7.0
4,その他/不明,14,6.0,11.0,6.0,5.0,5.0


In [12]:
by_price = error_analysis.group_summary(merged, "price_band")
by_area = error_analysis.group_summary(merged, "site_area_band")
print("=== 価格帯別 ===")
display(by_price.round(0))
print("=== 地積帯別 ===")
display(by_area.round(0))


=== 価格帯別 ===


,price_band,n,mae,rmse,bias,median_abs_pct_error,mean_abs_pct_error
0,>100k,5,5423.0,5962.0,-5423.0,5.0,4.0
1,30k-100k,33,4205.0,6224.0,-4181.0,7.0,10.0
2,10k-30k,158,1550.0,2026.0,-1543.0,10.0,10.0
3,5k-10k,249,510.0,783.0,-461.0,6.0,7.0
4,1k-5k,647,91.0,220.0,-53.0,2.0,3.0
5,<1k,188,76.0,393.0,65.0,3.0,16.0


=== 地積帯別 ===


,site_area_band,n,mae,rmse,bias,median_abs_pct_error,mean_abs_pct_error
0,<=100,166,991.0,2745.0,-975.0,4.0,7.0
1,300-500,90,678.0,1524.0,-626.0,6.0,9.0
2,500-1k,53,540.0,945.0,-295.0,6.0,13.0
3,100-150,427,432.0,1088.0,-402.0,2.0,4.0
4,150-200,284,362.0,847.0,-289.0,3.0,9.0
5,200-300,190,339.0,726.0,-297.0,3.0,7.0
6,1k-5k,58,81.0,140.0,-42.0,4.0,8.0
7,>5k,12,21.0,44.0,12.0,5.0,5.0


In [13]:
worst = error_analysis.top_errors(merged, n=10)
display(worst[["municipality", "lot_number", "use_district", "site_area", "far",
               "station_distance", "y_true", "y_pred", "abs_error", "pct_error"]].round(0))


,municipality,lot_number,use_district,site_area,far,station_distance,y_true,y_pred,abs_error,pct_error
0,港区,浜松町２丁目１０７番８,商業,89,600,240,30899.0,3248.0,27650.0,-89.0
1,目黒区,目黒本町５丁目２２番３外,近商,86,300,230,15814.0,3248.0,12566.0,-79.0
2,中央区,銀座２丁目２番１９外,商業,353,800,0,140510.0,132068.0,8442.0,-6.0
3,新宿区,愛住町７番１２,２中専,102,300,190,11569.0,3248.0,8320.0,-72.0
4,中央区,銀座４丁目２番４,商業,454,800,0,147797.0,139923.0,7874.0,-5.0
5,渋谷区,道玄坂１丁目１６番１８,商業,78,800,230,99359.0,91942.0,7417.0,-7.0
6,港区,新橋１丁目３０番９,商業,126,800,260,65714.0,59083.0,6631.0,-10.0
7,中野区,中野５丁目２１番７,商業,129,600,250,41938.0,35570.0,6368.0,-15.0
8,杉並区,上荻１丁目７番４外,商業,95,600,0,52421.0,46089.0,6332.0,-12.0
9,大田区,南蒲田３丁目１４６１番６,近商,107,400,150,9533.0,3248.0,6284.0,-66.0


## 8. 仮説の検証


In [14]:
hypotheses = {
    "容積率プレミアム": error_analysis.check_far_premium_hypothesis(merged),
    "小規模地": error_analysis.check_small_site_hypothesis(merged),
    "駅距離": error_analysis.check_station_distance_hypothesis(merged),
}
for name, values in hypotheses.items():
    print(f"=== {name} ===")
    for key, value in values.items():
        print(f"  {key:52s} {value}")
    print()


=== 容積率プレミアム ===
  spearman_far_vs_price_all                            0.6079659174205116
  spearman_far_vs_price_commercial                     0.5628101927490349
  n_high_far                                           221
  mean_bias_high_far                                   -1636.9964713431295
  mae_high_far                                         1682.4146317100658
  mae_all                                              477.14028464082037

=== 小規模地 ===
  n_small                                              44
  mae_small                                            1125.0117180021973
  mae_rest                                             454.07690028167747
  mean_bias_small                                      -1108.7000149063326
  price_ratio_small_vs_rest                            3.0601523525018672

=== 駅距離 ===
  pooled_spearman_distance_vs_residual                 0.5741984862700446
  pooled_spearman_distance_vs_price                    -0.6030939892564575
  median_distance_comm

### 検証の結論

上記の数値は `reports/error_analysis.md` に自動で書き出されます。
実際に確かめられたこと・確かめられなかったことはそちらにまとめています
（**支持された仮説と、支持されなかった仮説の両方**を記載しています）。


## 9. 統制実験: リーク列とランダム分割は何を隠すか

同じモデル（LightGBM）・同じテスト行で、**入力列だけを変えて**比較します。


## 9. 統制実験: リーク列とランダム分割は何を隠すか

同じモデル（LightGBM）・同じテスト行で、**入力列だけを変えて**比較します。


In [15]:
from src.models import build_estimator

# 予測年（令和8年）を、下の3設定すべてで同じ train/test 行になるように分割する。
# `test` は時系列分割のテスト側で、実測ラグが結合済み。ここからランダムに
# 半分を取ることで、時系列分割と同じデータを使いつつ分割方法だけを変えられる。
rand_train, rand_test = evaluate.random_split(test, cfg)
print(f"ランダム分割: train {len(rand_train)} / test {len(rand_test)} (同じ年・別地点)")
print(f"時系列分割  : train {len(train)} (令和7年) / test {len(test)} (令和8年)")
print()

settings = {}
for label, lag, leak in (
    ("属性のみ", False, False),
    ("属性 + 前年単価(lag)", True, False),
    ("属性 + lag + リーク列", True, True),
):
    columns, _ = features.resolve_feature_columns(
        rand_train, [rand_test], cfg, include_lag=lag, include_leakage=leak
    )
    design = features.build_design_matrix(rand_train, cfg, columns=columns)
    estimator = build_estimator("lightgbm", cfg, design, use_log_target=False)
    estimator.fit(rand_train[columns], rand_train[cfg.target_column])
    settings[label] = {
        "n_features": len(columns),
        "train MAE": evaluate.regression_metrics(rand_train[cfg.target_column], estimator.predict(rand_train[columns]))["mae"],
        "test MAE": evaluate.regression_metrics(rand_test[cfg.target_column], estimator.predict(rand_test[columns]))["mae"],
        "test R2": evaluate.regression_metrics(rand_test[cfg.target_column], estimator.predict(rand_test[columns]))["r2"],
    }

# 参考: 本番の時系列分割
settings["（参考）時系列分割・lagあり"] = {
    "n_features": len(runs["lightgbm"].feature_names),
    "train MAE": runs["lightgbm"].metrics["train"]["mae"],
    "test MAE": runs["lightgbm"].metrics["test"]["mae"],
    "test R2": runs["lightgbm"].metrics["test"]["r2"],
}
pd.DataFrame(settings).T.round(3)


ランダム分割: train 640 / test 640 (同じ年・別地点)
時系列分割  : train 2560 (令和7年) / test 1280 (令和8年)



,n_features,train MAE,test MAE,test R2
属性のみ,23.0,1341.014,3180.501,0.457
属性 + 前年単価(lag),24.0,865.407,1477.495,0.702
属性 + lag + リーク列,26.0,781.600,1530.664,0.723
（参考）時系列分割・lagあり,24.0,382.823,1347.595,0.914


### この表がこのプロジェクトの結論です

1. **リーク列を入れると test MAE が劇的に下がる。** しかしこれは予測精度ではなく、
   目的変数から直接作られた列を入力に加えた結果です。実運用では
   「翌年の価格」を知るために「翌年の前年価格」が必要、という循環になります。
2. **ランダム分割はリーク列なしでもスコアを甘く見せる。** 同じ年の別地点を当てるのは
   補間（interpolation）であり、翌年を当てる外挿（extrapolation）とは別のタスクです。
3. 本プロジェクトが報告する性能は、**必ず時系列分割の数値**です。

### 改善の試行

`reports/improvements.md` に、ラグ特徴量の追加・対数変換・別系列での再学習を
**前後比較**でまとめています（改善しなかった試行も正直に記載）。


In [16]:
# 生成物の確認（CLI パイプラインの出力と一致するはず）
metrics_path = cfg.reports_dir / "metrics.json"
if metrics_path.exists():
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    stored = payload["results"]["runs"]["lightgbm"]["metrics"]["test"]
    live = runs["lightgbm"].metrics["test"]
    print("reports/metrics.json の lightgbm test 指標 vs この notebook の計算:")
    for key in ("mae", "rmse", "r2"):
        print(f"  {key:5s} json={stored[key]:12,.4f}  notebook={live[key]:12,.4f}  "
              f"一致={'OK' if abs(stored[key]-live[key]) < 1e-6 else 'NG'}")
else:
    print("reports/metrics.json が未生成です。先に `python -m src.pipeline --stage all` を実行してください。")


reports/metrics.json が未生成です。先に `python -m src.pipeline --stage all` を実行してください。


---
### まとめ（面接で説明する 3 点）

1. **目的変数の設計**: 総額ではなく平米単価にした。総額では「広いほど高い」という
   自明な関係を当てるだけになる。
2. **リークの実測**: 前年価格列は目的変数と 0.95 の相関を持ち、
   入れるとスコアが跳ね上がる。時系列分割と特徴量ホワイトリストで構造的に排除し、
   その効果を統制実験として定量化した。
3. **正直な結論**: この課題の最強ベースラインは「前年単価をそのまま使う」（MAE 約 750 円/㎡）で、
   勾配ブースティング（MAE 約 1,300 円/㎡）はそれを超えられない。
   一方で属性のみの線形モデル（MAE 約 1,600 円/㎡）は明確に改善しており、
   **「機械学習が効く範囲」と「効かない範囲」を切り分けた**ことが成果です。
